# RobBERT v2 paired fine-tuning

"
        "This GPU experiment compares two candidates under one frozen project contract:

"
        "1. **RobBERT Logistic** — end-to-end encoder fine-tuning with weighted three-way softmax.
"
        "2. **RobBERT Ordinal** — end-to-end encoder fine-tuning with a rank-consistent CORAL head.

"
        "The notebook is deliberately thin: data preparation, hashes, objectives, metrics, and bundle "
        "validation live in `dutch_sentiment`. The 960-row test split is never used for checkpoint selection.

## Setup

Upload `robbert_colab_source.zip` when prompted. It contains the current local source snapshot and the immutable assignment CSV; no secrets or local MLflow database are included.

In [ ]:
from google.colab import drive, files
"
        "drive.mount('/content/drive')
"
        "uploaded = files.upload()
"
        "assert 'robbert_colab_source.zip' in uploaded, 'Upload robbert_colab_source.zip'

In [ ]:
import shutil
"
        "from pathlib import Path

"
        "workspace = Path('/content/robbert_source')
"
        "if workspace.exists():
"
        "    shutil.rmtree(workspace)
"
        "shutil.unpack_archive('/content/robbert_colab_source.zip', workspace)
"
        "%cd /content/robbert_source
"
        "%pip install -q --ignore-requires-python -e '.[train,finetune]'

## Checks

Confirm that Colab assigned a CUDA GPU and that the checked-in model revision and immutable split contract are visible before training.

In [ ]:
import torch, yaml
"
        "from pathlib import Path

"
        "assert torch.cuda.is_available(), 'Select Runtime → Change runtime type → T4 GPU'
"
        "config = yaml.safe_load(Path('configs/models/robbert_v2.yaml').read_text())
"
        "print('GPU:', torch.cuda.get_device_name(0))
"
        "print('Model:', config['model']['model_id'])
"
        "print('Revision:', config['model']['revision'])
"
        "print('Candidates:', config['candidates'])

## Run paired experiment

Each completed file is written to Google Drive. If the free runtime disconnects after completion, the bundle remains available there.

In [ ]:
from pathlib import Path

"
        "output_dir = Path('/content/drive/MyDrive/Workspace365_assignment/robbert_v2')
"
        "output_dir.mkdir(parents=True, exist_ok=True)
"
        "!sentiment-robbert-finetune --config configs/models/robbert_v2.yaml --output-dir "{output_dir}"

## Verify and summarize

Revalidate all checksums and display only the decision metrics needed for the paired comparison.

In [ ]:
import json
"
        "from dutch_sentiment.experiments.robbert_finetune import verify_bundle

"
        "result = verify_bundle(output_dir, 'configs/models/robbert_v2.yaml')
"
        "for candidate in result['candidates']:
"
        "    metrics = candidate['test_metrics']
"
        "    negative = metrics['per_class']['Negative']
"
        "    print({
"
        "        'objective': candidate['objective'],
"
        "        'macro_f1': round(metrics['macro_f1'], 4),
"
        "        'accuracy': round(metrics['accuracy'], 4),
"
        "        'negative_precision': round(negative['precision'], 4),
"
        "        'negative_recall': round(negative['recall'], 4),
"
        "        'ordinal_mae': round(metrics['ordinal_mae'], 4),
"
        "        'severe_error_rate': round(metrics['severe_error_rate'], 4),
"
        "    })

## Handoff

Download or retain `/MyDrive/Workspace365_assignment/robbert_v2`. Back in the local repository, run:

```bash
make robbert-verify BUNDLE=/path/to/robbert_v2
make robbert-import BUNDLE=/path/to/robbert_v2
```

Importing is intentionally local so Colab never writes the SQLite MLflow store directly.